# Review reference and *EOlife X* measurements

### Overview
This notebook provides an **interactive interface** for reviewing the reference and values extracted from *EOlife X*'s display used in the publication on **the validation of *EOlife X***.  
The dataset originates from a **porcine model of cardiopulmonary resuscitation (CPR)**.

### Usage Instructions
- Use the **dropdown menu** to select the case to be reviewed.  
- The corresponding physiological signals and annotations are displayed in synchronized subplots, with an **overview of the full duration** shown beneath.  
  - The **overview plot is clickable**: clicking a region updates the main display.  
  - The **highlighted box** in the overview indicates the **currently visible time window** in the main plots.  
- **Navigation controls:**  
  - Press **`+`** or **`-`** to zoom in and out along the time axis.  
  - Use the **arrow keys** to pan left or right.  
  - Hover the mouse over a subplot and scroll the **mouse wheel** to adjust the **vertical scaling** of that trace.  

### Notes
This interactive viewer facilitates **visual inspection and validation** of value pairs obtained from raw experimental data.  
It serves as a supplementary tool to enhance **transparency**, and  **reproducibility**, and **understanding** of the analyses presented in the publication.

In [1]:
from vitabel import Vitals
from pathlib import Path
import pandas as pd
from copy import deepcopy
import ipywidgets as widgets
from IPython.display import display

def plot_curves(case: Vitals, start: pd.Timestamp | None = None, stop : pd.Timestamp| None = None):
    id=case.metadata["case_index"]

    case.get_label("Inspiration Begin").plotstyle.update(linestyle="dotted")
    case.get_label("Expiration Begin").plotstyle.update(linestyle="dashed")

    #invisible channel to get resp rate ylim down to zero   
    ghost_chan = deepcopy(case.get_channel("f"))
    ghost_chan.rename("ghost")
    ghost_chan.data[:] = 0
    ghost_chan.plotstyle.update(marker="", label="", alpha=0)
    case.add_channel(ghost_chan)
        
    v_exp = case.get_channels("Expiratory Volume")
    v_insp = case.get_channels("Inspiratory Volume")
    for chan in v_exp + v_insp:
        chan.plotstyle.update(alpha = 0.4)

    plot = case.plot_interactive(
        channels = [["Flow Interpolated"],
                    ["Pressure Interpolated"],
                    ["ve", "Expiratory Volume"], 
                    ["vi", "Inspiratory Volume"],
                    ["f", "ghost"]],
        labels=[["Inspiration", "Expiration"],
                ["Inspiration", "Expiration"],
                ["Inspiration Begin", "Expiration Begin", "VTexp", "exp_gt" ],
                ["Inspiration Begin", "Expiration Begin",  "VTinsp", "insp_gt"],
                ["Inspiration Begin", "Expiration Begin", "Respiratory Rate", "rr_corr_gt", "f_gt"]],
        channel_overviews=[["Inspiratory Volume"]],
        start = start,
        stop = stop,
        subplots_kwargs={
        "figsize": (18, 10), #7.48,
        "dpi": 100,
        "gridspec_kw": {"height_ratios": [1,1,1,1,1,0.5]}
        },
        time_unit="s")
 
    fig = plot.center.figure
    axes = fig.get_axes()
        
    font_size=10
    axes[0].set_ylabel(r"Flow", fontsize=font_size)
    axes[1].set_ylabel(r"Pressure", fontsize=font_size)
    axes[2].set_ylabel(r"$Vt_{\mathrm{e}}$", fontsize=font_size)
    axes[3].set_ylabel(r"$Vt_{\mathrm{i}}$", fontsize=font_size)
    axes[4].set_ylabel(r"$f$", fontsize=font_size)
    axes[5].set_ylabel("Overview")

    # Define units for each subplot
    units = ["L/min", "cmH₂O", "mL", "mL", " min⁻¹"]
    
    for i, unit in enumerate(units):
        # Unit label close to right edge
        axes[i].text(1.02, 0.5, unit,
                     transform=axes[i].transAxes,
                     fontsize=font_size - 1,
                     color='grey',
                     ha='left',
                     va='center',
                     rotation=90)

    for ax in axes:
        legend = ax.get_legend()
        if legend:
            legend.remove()
            
    fig.tight_layout(rect=[0, 0.015, 0.97, 1])  # Leaves space at the top for the suptitle
    fig.subplots_adjust(hspace=0)

    for ax in axes[:-2]:
        ax.set_xlabel(None)
        ax.set_xticklabels([])  # Remove x-tick labels

    # Shift the last subplot down to create a gap
    pos = axes[5].get_position()
    axes[5].set_position([pos.x0, pos.y0 - 0.03, pos.width, pos.height])
    
    fig.suptitle("")
    
    return plot

# Load Data and Generate Plots
cases = {}
widgets_map = {}
for case_path in Path("./data/").glob("*.json"):
    case_id = case_path.stem
    case = Vitals()
    case.load_data(case_path) 
    cases[case_id]=case
    widgets_map[case_id]=plot_curves(case)

# Generated a Dropdown Menus to swtich betweeen cases
# 1) Dropdown with keys
dd = widgets.Dropdown(
    options=list(widgets_map.keys()),
    value=next(iter(widgets_map)),  # first key as default
    description="Select Case:",
    layout=widgets.Layout(width="180px")
)
# 2) A container that will hold exactly one widget (the chosen AppLayout)
holder = widgets.Box()
# 3) Callback: swap the child on selection
def on_select(change):
    if change["name"] == "value":
        selected = change["new"]
        holder.children = (widgets_map[selected],)
dd.observe(on_select, names="value")
# 4) Initialize view and display
holder.children = (widgets_map[dd.value],)
spacer = widgets.HTML("<br>")  # two line breaks
display(dd, spacer, holder)

Dropdown(description='Select Case:', layout=Layout(width='180px'), options=('221122', '221117', '230110', '221…

HTML(value='<br>')

Box(children=(AppLayout(children=(Tab(children=(VBox(children=(HTML(value='\n                    <p>\n        …